
# mT5 small — Baseline TF Training (Without Wuxia Domain)

**TFG Hugo Silvosa – Baseline NMT (mt5-small)**  
Este cuaderno  evalúa un modelo **MarianMT** (`google/mt5-small`)sin entrenar en dominio usando un dataset de **wuxia** (chino->inglés) ya preparado en formato `datasets` (HF).





## 1) Entorno de ejecución e instalación de dependencias

In [1]:

import os, random, math
import numpy as np

import torch
print("CUDA disponible:", torch.cuda.is_available())
print("Número de GPUs:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("Nombre de la GPU:", torch.cuda.get_device_name(0))



CUDA disponible: True
Número de GPUs: 1
Nombre de la GPU: NVIDIA GeForce RTX 3060


> **Requisitos del dataset**: directorio HF Datasets con *splits* `train`, `validation`, `test` y columnas `zh` (chino) y `en` (inglés):  
> `processed_data/wuxia_zh_en_clean/`


In [8]:
# Configuración de carpetas para entorno LOCAL
from pathlib import Path
BASE_DIR = Path.cwd().parent.parent.parent.parent.parent
BASE_DIR.mkdir(exist_ok=True)

for sub in ["evaluation", "models", "processed_data"]:
    (BASE_DIR / sub).mkdir(parents=True, exist_ok=True)

print("Base:", BASE_DIR.resolve())
print("Estructura creada (si no existía):")
for p in ["evaluation", "models", "proccesed data"]:
    print(" -", (BASE_DIR / p).resolve())

# Nota: el dataset debe existir en: CORPUS/proccesed data/wuxia_zh_en_clean


Base: C:\Users\Usuario\Desktop\TFG\CORPUS
Estructura creada (si no existía):
 - C:\Users\Usuario\Desktop\TFG\CORPUS\evaluation
 - C:\Users\Usuario\Desktop\TFG\CORPUS\models
 - C:\Users\Usuario\Desktop\TFG\CORPUS\proccesed data


## 2) Configuración

In [9]:
from dataclasses import dataclass

@dataclass
class Config:
    # Rutas (local)
    dataset_dir: Path  = BASE_DIR / "processed_data" / "wuxia_zh_en_clean"   
    output_dir: Path   = BASE_DIR / "models" / "pretrain_mt5_small"             
    ckpt_dir: Path     = BASE_DIR / "checkpoints"
    training_dir: Path = BASE_DIR / "training"
    evaluation_dir: Path = BASE_DIR / "evaluation"
    translate_dir: Path = BASE_DIR / "evaluation" / "translate"
    translate_file: Path =   "pre_mt5.txt"
    results_file: Path = "pre_results.txt"
    # Columnas del dataset
    src_col: str = "zh"
    tgt_col: str = "en"

    # Modelo 
    model_ckpt: str = "google/mt5-small"

    # Prefijo de instrucción para T5/mT5 (para evitar <extra_id_0>)
    use_instruction_prefix: bool = True
    translation_prefix: str = "translate Chinese to English: "

    # Entrenamiento
    seed: int = 42
    max_source_length: int = 128
    max_target_length: int = 128
    batch_size: int = 16
    epochs: int = 10
    learning_rate: float = 2e-5
    weight_decay: float = 0.01
    early_stopping_patience: int = 3
    fraction: float = 1

cfg = Config()
print(cfg)


Config(dataset_dir=WindowsPath('c:/Users/Usuario/Desktop/TFG/CORPUS/processed_data/wuxia_zh_en_clean'), output_dir=WindowsPath('c:/Users/Usuario/Desktop/TFG/CORPUS/models/pretrain_mt5_small'), ckpt_dir=WindowsPath('c:/Users/Usuario/Desktop/TFG/CORPUS/checkpoints'), training_dir=WindowsPath('c:/Users/Usuario/Desktop/TFG/CORPUS/training'), evaluation_dir=WindowsPath('c:/Users/Usuario/Desktop/TFG/CORPUS/evaluation'), translate_dir=WindowsPath('c:/Users/Usuario/Desktop/TFG/CORPUS/evaluation/translate'), translate_file='pre_mt5.txt', results_file='pre_results.txt', src_col='zh', tgt_col='en', model_ckpt='google/mt5-small', use_instruction_prefix=True, translation_prefix='translate Chinese to English: ', seed=42, max_source_length=128, max_target_length=128, batch_size=16, epochs=10, learning_rate=2e-05, weight_decay=0.01, early_stopping_patience=3, fraction=1)


In [10]:
import random, numpy as np, os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Usando dispositivo:", device)



# Semillas para reproducibilidad
torch.manual_seed(cfg.seed)
np.random.seed(cfg.seed)
random.seed(cfg.seed)
os.environ["PYTHONHASHSEED"] = str(cfg.seed)


if device.type == "cuda":
    torch.cuda.manual_seed_all(cfg.seed)
    # Para reproducibilidad estricta (ligera penalización de rendimiento)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print("Semillas fijadas y backend configurado.")


Usando dispositivo: cuda
Semillas fijadas y backend configurado.


## 4) Cargar dataset (Hugging Face Datasets)

In [11]:

from datasets import load_from_disk, DatasetDict

assert os.path.isdir(cfg.dataset_dir), f"No se encuentra el dataset en: {cfg.dataset_dir}"
raw_ds: DatasetDict = load_from_disk(cfg.dataset_dir)
print(raw_ds)

# Validar columnas
def _check_cols(ds, src_col, tgt_col, split):
    cols = ds.column_names
    assert src_col in cols and tgt_col in cols, f"El split '{split}' debe contener columnas '{src_col}' y '{tgt_col}'. Columnas: {cols}"

for split in ["train", "validation", "test"]:
    assert split in raw_ds, f"Falta el split '{split}' en el dataset."
    _check_cols(raw_ds[split], cfg.src_col, cfg.tgt_col, split)

#  pruebas rápidas
def take_fraction(ds, frac, seed=42):
    if frac >= 1.0:
        return ds
    n = max(1, int(len(ds) * frac))
    return ds.shuffle(seed=seed).select(range(n))

train_ds = take_fraction(raw_ds["train"], cfg.fraction, seed=cfg.seed)
val_ds   = take_fraction(raw_ds["validation"], cfg.fraction, seed=cfg.seed)
test_ds  = take_fraction(raw_ds["test"], cfg.fraction, seed=cfg.seed)

print(train_ds[:2])
print(f"Tam. train/val/test (fracción={cfg.fraction}):", len(train_ds), len(val_ds), len(test_ds))


DatasetDict({
    train: Dataset({
        features: ['zh', 'en'],
        num_rows: 417208
    })
    validation: Dataset({
        features: ['zh', 'en'],
        num_rows: 52151
    })
    test: Dataset({
        features: ['zh', 'en'],
        num_rows: 52151
    })
})
{'zh': ['第章 听到白小纯的话语，看到圣皇的迟疑，邪皇这里顿时呼吸一促，他目中刹那就露出凌厉之芒，右手猛的一挥，顿时那残破的红日，骤然幻化', '尤其是看到人群内的宋缺时，神算子立刻警惕，他当年在空城，是第一个跟随白小纯的，受到了重用，如今却成为了第二个，他顿时就视宋缺为竞争对手'], 'en': ['chapter- The Saint-Emperor hesitated, and the Vile-Emperor sucked in a breath, eyes flickering with cold light as he waved his hand to summon his damaged red sun', 'That was even more the case when he noticed Song Que among Bai Xiaochun’s men, which immediately got him even more on guard. Back in Sky City, Master God-Diviner had been the first to start following Bai Xiaochun again, and it had led to incredible benefits. Now, he was only the second to join him, which put Song Que in his sights as a major rival']}
Tam. train/val/test (fracción=1): 417208 52151 52151

## 4) Cargar tokenizador y modelo 

In [12]:

from transformers import MT5Tokenizer, MT5ForConditionalGeneration

# Tokenizer
tokenizer = MT5Tokenizer.from_pretrained(cfg.model_ckpt)

# Modelo
model = MT5ForConditionalGeneration.from_pretrained(cfg.model_ckpt)

#  Salvaguardas de tokens especiales
# Pad token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

# Decoder start token
if model.config.decoder_start_token_id is None:
    model.config.decoder_start_token_id = tokenizer.pad_token_id

# EOS token
if model.config.eos_token_id is None:
    model.config.eos_token_id = tokenizer.eos_token_id

# Evitar warnings con gradient checkpointing
model.config.use_cache = False

# Enviar a dispositivo
model.to(device)

# Info
print("Modelo:", cfg.model_ckpt)
print("pad_token_id:", tokenizer.pad_token_id)
print("eos_token_id:", tokenizer.eos_token_id)
print("decoder_start_token_id:", model.config.decoder_start_token_id)


The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'T5Tokenizer'. 
The class this function is called from is 'MT5Tokenizer'.


Modelo: google/mt5-small
pad_token_id: 0
eos_token_id: 1
decoder_start_token_id: 0


## 5) Preprocesamiento y tokenización

In [13]:

max_source_length = cfg.max_source_length
max_target_length = cfg.max_target_length

def preprocess_function(examples):
    #  Source (ZH) con prefijo de instrucción para mT5
    if cfg.use_instruction_prefix:
        src_texts = [cfg.translation_prefix + s.strip() for s in examples[cfg.src_col]]
    else:
        src_texts = [s.strip() for s in examples[cfg.src_col]]

    #  Target (EN)
    tgt_texts = [t.strip() for t in examples[cfg.tgt_col]]

    #  Tokenización: 
    model_inputs = tokenizer(
        src_texts,
        max_length=max_source_length,
        truncation=True,
        padding=False
    )
    labels = tokenizer(
        text_target=tgt_texts,           # para leer labels bien
        max_length=max_target_length,
        truncation=True,
        padding=False
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_datasets = raw_ds.map(
    preprocess_function,
    batched=True,
    remove_columns=raw_ds["train"].column_names
)

train_ds = take_fraction(tokenized_datasets["train"], cfg.fraction, seed=cfg.seed)
val_ds   = take_fraction(tokenized_datasets["validation"], cfg.fraction, seed=cfg.seed)
test_ds  = take_fraction(tokenized_datasets["test"], cfg.fraction, seed=cfg.seed) if "test" in tokenized_datasets else val_ds

print("Ejemplo tokenizado:", {k: type(v) for k,v in train_ds[0].items()})


Map:   0%|          | 0/417208 [00:00<?, ? examples/s]

Map:   0%|          | 0/52151 [00:00<?, ? examples/s]

Map:   0%|          | 0/52151 [00:00<?, ? examples/s]

Ejemplo tokenizado: {'input_ids': <class 'list'>, 'attention_mask': <class 'list'>, 'labels': <class 'list'>}


## 6) Evaluación 

In [14]:

from tqdm.auto import tqdm
import sacrebleu
from sacrebleu.metrics import CHRF, TER
from rouge_score import rouge_scorer
import nltk
from nltk.translate.meteor_score import meteor_score
from nltk.tokenize import wordpunct_tokenize
import numpy as np
import torch
import json
import time

start = time.time()
# Descargar recursos de NLTK (para METEOR)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

#  Parámetros 
EVAL_MAX_SAMPLES = 1000        # None = todo el split
PRED_BEAMS = 4
BATCH_EVAL = max(1, cfg.batch_size // 2)

#  Comprobaciones 
assert 'model' in globals(), "No se encontró `model`. Carga el modelo antes."
assert 'tokenizer' in globals(), "No se encontró `tokenizer`. Cárgalo antes."
assert 'val_ds' in globals() and 'test_ds' in globals(), "Faltan `val_ds` y/o `test_ds`."
assert 'cfg' in globals(), "Falta `cfg`."



# Asegurar pad_token_id
if tokenizer.pad_token_id is None and tokenizer.eos_token_id is not None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

# mT5: asegurar decoder_start_token_id
if getattr(model.config, 'decoder_start_token_id', None) is None and tokenizer.pad_token_id is not None:
    model.config.decoder_start_token_id = tokenizer.pad_token_id


#  Seleccionar split 
eval_raw = test_ds if len(test_ds) > 0 else val_ds
n_total = len(eval_raw)
n_eval = n_total if (EVAL_MAX_SAMPLES is None) else min(n_total, int(EVAL_MAX_SAMPLES))
assert n_eval > 0, "No hay ejemplos para evaluar."

def decode_ids_to_text(dataset, id_col):
    return [
        tokenizer.decode(ids, skip_special_tokens=True)
        for ids in dataset[id_col]
    ]

src_texts = decode_ids_to_text(eval_raw, "input_ids")[:n_eval]
ref_texts = decode_ids_to_text(eval_raw, "labels")[:n_eval]

#  mT5: asegurar prefijo de instrucción en la entrada 
if hasattr(cfg, 'use_instruction_prefix') and cfg.use_instruction_prefix:
    pref = getattr(cfg, 'translation_prefix', 'translate Chinese to English: ')
    def _ensure_pref(s):
        return s if s.startswith(pref) else (pref + s)
    src_texts = [_ensure_pref(s) for s in src_texts]

#  Generación por lotes 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def batched_generate(texts, batch_size=8, max_length=128, num_beams=4):
    preds = []
    model.eval()
    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size)):
            batch = texts[i:i+batch_size]
            inputs = tokenizer(
                batch,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=cfg.max_source_length
            ).to(device)
            outputs = model.generate(
                **inputs,
                max_length=max_length,
                num_beams=num_beams,
                early_stopping=True
            )
            preds.extend(tokenizer.batch_decode(outputs, skip_special_tokens=True))
    return preds

preds = batched_generate(
    src_texts,
    batch_size=BATCH_EVAL,
    max_length=cfg.max_target_length,
    num_beams=PRED_BEAMS
)

#  Métricas 
bleu_corpus = sacrebleu.corpus_bleu(preds, [ref_texts]).score

chrf_metric = CHRF(word_order=2)
chrf_corpus = chrf_metric.corpus_score(preds, [ref_texts]).score

ter_metric = TER()
ter_corpus = ter_metric.corpus_score(preds, [ref_texts]).score

def compute_rougeL_f1(hyp_list, ref_list):
    scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
    f1s = []
    for h, r in zip(hyp_list, ref_list):
        scores = scorer.score(r, h)
        f1s.append(scores['rougeL'].fmeasure * 100.0)
    return float(np.mean(f1s))

rougeL_f1 = compute_rougeL_f1(preds, ref_texts)

def compute_meteor(hyp_list, ref_list):
    scores = []
    for h, r in zip(hyp_list, ref_list):
        hyp_tok = wordpunct_tokenize(h)
        ref_tok = wordpunct_tokenize(r)
        scores.append(meteor_score([ref_tok], hyp_tok))
    return float(np.mean(scores)) * 100.0

meteor_avg = compute_meteor(preds, ref_texts)

end_time = time.time()

results = {
    "model" : cfg.model_ckpt,
    "n_eval": n_eval,
    "num_beams": PRED_BEAMS,
    "batch_eval": BATCH_EVAL,
    "sacrebleu": round(bleu_corpus, 4),
    "chrf2": round(chrf_corpus, 4),
    "ter": round(ter_corpus, 4),
    "rougeL_f1": round(rougeL_f1, 4),
    "meteor": round(meteor_avg, 4), 
    "execution_time": round(end_time - start, 2)
}

os.makedirs(cfg.evaluation_dir, exist_ok=True)

res_file = os.path.join(cfg.evaluation_dir, cfg.results_file)

with open(res_file, "a", encoding="utf-8") as f:
    f.write("\n")
    f.write(json.dumps(results, ensure_ascii=False, indent=4))

print(results)



  0%|          | 0/125 [00:00<?, ?it/s]

{'model': 'google/mt5-small', 'n_eval': 1000, 'num_beams': 4, 'batch_eval': 8, 'sacrebleu': 0.0027, 'chrf2': 1.2654, 'ter': 100.0, 'rougeL_f1': 0.0, 'meteor': 0.0, 'execution_time': 59.88}


In [15]:
print(res_file)

c:\Users\Usuario\Desktop\TFG\CORPUS\evaluation\pre_results.txt


## 12) Muestra cualitativa (n ejemplos aleatorios)

In [ ]:
import random
import torch

# Mostrar predicciones aleatorias 
n_show = 100
idxs = random.sample(range(len(eval_raw)), k=min(n_show, len(eval_raw)))

model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

for i in idxs:
    # Decodificar texto fuente y referencia desde dataset tokenizado
    zh_in = tokenizer.decode(eval_raw[i]["input_ids"], skip_special_tokens=True)
    en_ref = tokenizer.decode(eval_raw[i]["labels"], skip_special_tokens=True)

    # Si el input ya contiene el prefijo, lo retiramos solo para imprimir "ZH:"
    if cfg.use_instruction_prefix and zh_in.startswith(cfg.translation_prefix):
        zh_print = zh_in[len(cfg.translation_prefix):]
    else:
        zh_print = zh_in

    # Preparar entrada para generate (siempre con prefijo)
    zh_for_gen = cfg.translation_prefix + zh_print if cfg.use_instruction_prefix else zh_print
    inputs = tokenizer([zh_for_gen], return_tensors="pt", padding=True, truncation=True, max_length=cfg.max_source_length)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_length=cfg.max_target_length,
            num_beams=4,
            early_stopping=True
        )

    en_pred = tokenizer.decode(out[0], skip_special_tokens=True)

    print("="*80)
    print("ZH:", zh_print)
    print("EN (ref):", en_ref)
    print("EN (pred):", en_pred)


ZH: 最关键的是——真正的天庭,不是在这里,而是在我们的心中
EN (ref): And most importantly, the true Heavenly Court is not here, it is within our hearts
EN (pred): <extra_id_0>。
ZH: 这一切他都清楚,可他还是脚步停顿下来,没有选择离去,因为 他不能错过这个机会,一旦错过,哪怕是在这炼魂壶的世界内,他想要去找到白浩,也都极为艰难,且一旦在这过程中,白浩的魂出现了灭亡,那么将是他一生的遗憾
EN (ref): He could not pass up this oppor tunity. After all, even knowing that Bai Hao’s soul was in the Necromancer Kettle wouldn’t necessarily do much good. Even going to search for him in that one particular area would be difficult. Furthermore, if Bai Hao’s soul were to die, then Bai Xiaochun would feel regret over the matter for the rest of his life
EN (pred): <extra_id_0>。
ZH: 我不信
EN (ref): I do not believe it
EN (pred): <extra_id_0>
ZH: 黄蓉心道
EN (ref): Huang Rong thought
EN (pred): <extra_id_0>
ZH: 只听后面蹄声急促,一骑马追来
EN (ref): The sound of hoof beats came as another horse chased up from behind
EN (pred): <extra_id_0>


In [ ]:

os.makedirs(cfg.translate_dir, exist_ok=True)
translate_path = os.path.join(cfg.translate_dir, cfg.translate_file)

with open(translate_path, "w", encoding="utf-8") as f:
    for i in range(len(eval_raw) // 2):
        zh_in = tokenizer.decode(eval_raw[i]["input_ids"], skip_special_tokens=True)
        en_ref = tokenizer.decode(eval_raw[i]["labels"], skip_special_tokens=True)

        # Si el input ya contiene el prefijo, lo retiramos solo para imprimir "ZH:"
        if cfg.use_instruction_prefix and zh_in.startswith(cfg.translation_prefix):
            zh_print = zh_in[len(cfg.translation_prefix):]
        else:
            zh_print = zh_in

        # Preparar entrada para generate siempre con prefijo (evitar <id_0> o salidas vacias)
        zh_for_gen = cfg.translation_prefix + zh_print if cfg.use_instruction_prefix else zh_print
        inputs = tokenizer([zh_for_gen], return_tensors="pt", padding=True, truncation=True, max_length=cfg.max_source_length)
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_length=cfg.max_target_length,
                num_beams=4,
                early_stopping=True
            )

        en_pred = tokenizer.decode(out[0], skip_special_tokens=True)

        # Guardar en el archivo
        f.write("="*80 + "\n")
        f.write("ZH: " + zh_in + "\n")
        f.write("EN (ref): " + en_ref + "\n")
        f.write("EN (pred): " + en_pred + "\n\n")
